In [ ]:
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd
import seaborn as sns
import os
import glob
from datetime import datetime
from datetime import timedelta
from matplotlib import pyplot as plt
import matplotlib.dates as md
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import warnings
from matplotlib import cm
import matplotlib.dates as Ntes
from scipy.interpolate import interp2d
warnings.filterwarnings('ignore')
#import datetime
import scipy.ndimage as ndimage
from matplotlib import cm
import geopy.distance
#import matplotlib as mpl
from scipy.interpolate import interp1d
from sklearn.linear_model import LinearRegression
from shapely.geometry import Point
import geopandas as gpd
from geopandas import GeoDataFrame
import leafmap
import plotly.express as px
import matplotlib as mpl
import xarray as xr
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D
#import pysplit
import netCDF4
import xarray as xr

In [ ]:
csv_file_path = 'C:/Users/taiwoajayi/\OneDrive - University of Arizona/Arizona_ozone/Ozone/No2full_data.csv'

# Read the CSV file into a pandas DataFrame
NO2 = pd.read_csv(csv_file_path, sep = ',', skiprows=0)
NO2['Timestamp'] = pd.to_datetime(NO2['Date Local'] + ' ' + NO2['Time Local'])
NO2['Sample Measurement'] = NO2.apply(lambda row: row['MDL'] * 0.5 * row['Sample Measurement']  if row['Sample Measurement'] < row['MDL'] else row['Sample Measurement'], axis=1)
# Define the columns of interest
columns_of_interest = ['Date Local', 'State Code', 'County Code', 'Site Num', 'Latitude', 'Longitude', 'Timestamp', 'Sample Measurement', 'MDL']

NA_2 = NO2[(NO2['Timestamp'].dt.year >= 2010) & (NO2['Timestamp'].dt.year <= 2022)]


# Extract the columns from N_03
#N_O = NO2[columns_of_interest]
#N_O.rename(columns={'Sample Measurement': 'sample'}, inplace=True)

NO2

In [ ]:
# Define bounding boxes (min_lon, max_lon, min_lat, max_lat)
locations = {
    "Children's Park": (-110.9823, -110.9823, 32.29515, 32.29515),
    "Craycroft": (-110.878067, -110.878067, 32.204411, 32.204411)
}

# Extract data from bounding boxes
def extract_locations(data, locations):
    extracted_data = {}
    for location, (min_long, max_long, min_lat, max_lat) in locations.items():
        data_location = data[
            (data['Longitude'] >= min_long) & (data['Longitude'] <= max_long) &
            (data['Latitude'] >= min_lat) & (data['Latitude'] <= max_lat)
        ][['Latitude', 'Longitude', 'Timestamp', 'Sample Measurement', 'MDL']].copy()
        extracted_data[location] = data_location
    return extracted_data

# Calculate yearly medians from 2010–2022
def yearly_median_data(data):
    data = data[(data['Timestamp'].dt.year >= 2010) & (data['Timestamp'].dt.year <= 2022)]
    data['Year'] = data['Timestamp'].dt.year
    yearly = data.groupby('Year')['Sample Measurement'].median().reset_index()
    return yearly

# Example usage (assuming your DataFrame is named N_2021)
extracted_data = extract_locations(NO2, locations)
yearly_medians = {loc: yearly_median_data(df) for loc, df in extracted_data.items()}

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
for location, df in yearly_medians.items():
    if not df.empty:
        ax.plot(df['Year'], df['Sample Measurement'], label=location, marker='o')
ax.tick_params(axis='both', labelsize=20)
ax.minorticks_on()
ax.tick_params(axis='x', which='minor', length=4, color='gray')
#ax.set_title('Yearly Median  (2010–2022)')
ax.set_ylabel('NO$_2$ (ppb)', fontsize=20)
ax.set_xlabel('Year', fontsize=20)
ax.grid(True)
ax.legend(fontsize=20)
plt.tight_layout()
plt.show()


In [ ]:
# Assuming location_df_daily is your DataFrame
'''condition = ~((NO2['Timestamp'] >= pd.to_datetime('2020-03-15')) & (NO2['Timestamp'] <= pd.to_datetime('2020-08-31')))
NO2 = NO2[condition]
NO2'''

In [ ]:
NA_2020 = NO2[(NO2['Timestamp'].dt.year >= 2020) & (NO2['Timestamp'].dt.year <= 2020)]
NA_2020

In [ ]:
N_2006 = NO2[(NO2['Timestamp'].dt.year >= 2006) & (NO2['Timestamp'].dt.year <= 2009)]
N_2006


In [ ]:
N_2011 = NO2[(NO2['Timestamp'].dt.year >= 2010) & (NO2['Timestamp'].dt.year <= 2019)]
N_2011


In [ ]:
#N_2016 = NO2[(NO2['Timestamp'].dt.year >= 2016) & (NO2['Timestamp'].dt.year <= 2019)]
#N_2016


In [ ]:
N_2021 = NO2[(NO2['Timestamp'].dt.year >= 2021) & (NO2['Timestamp'].dt.year <= 2022)]
N_2021

In [ ]:
# Define longitude and latitude ranges for each location
locations = {
    'Children\'s Park': (-110.9823, -110.9823, 32.29515, 32.29515),
    'Green Valley': (-110.99644, -110.99644, 31.87952,  31.87952),
    'Coach Line': (-111.12716, -111.12716, 32.38082, 32.38082),
    'Rose Elementary': (-110.980134, -110.980134, 32.172995,  32.172995),
    'Fairgrounds': (-110.774357, -110.774357, 32.04767,  32.04767),
    'Tangerine': (-111.06352, -111.06352, 32.425261,  32.425261),
    'Craycroft': (-110.878067, -110.878067, 32.204411,  32.204411),
    'Saguaro Park': (-110.737116, -110.737116, 32.174538,  32.174538)
}

# Define the function to extract location-based data
def extract_locations(data, locations):
    extracted_data = {}
    
    # Iterate through locations
    for location, (min_long, max_long, min_lat, max_lat) in locations.items():
        # Filter data for the location
        data_location = data[
            (data['Longitude'] >= min_long) & 
            (data['Longitude'] <= max_long) & 
            (data['Latitude'] >= min_lat) & 
            (data['Latitude'] <= max_lat)
        ]
        
        # Copy relevant columns from the original DataFrame
        data_location = data_location[['Latitude', 'Longitude', 'Timestamp', 'Sample Measurement', 'MDL']].copy()
        
        # Store the extracted data in the dictionary
        extracted_data[location] = data_location
    
    return extracted_data

# Define the function to calculate monthly averages
def monthly_average_data(data):
    # Add a 'Month' column to the data
    data['Month'] = data['Timestamp'].dt.month
    
    # Group data by month and calculate the mean for 'Sample Measurement', 'Latitude', and 'Longitude'
    monthly_avg = data.groupby('Month').agg({
        'Sample Measurement': 'mean',
        'Latitude': 'mean',
        'Longitude': 'mean'
    }).reset_index()
    
    # Rename columns for clarity
    monthly_avg.columns = ['Month', 'Sample Measurement', 'Latitude', 'Longitude']
    
    return monthly_avg

# Example usage
# Assuming `data` is a pandas DataFrame containing the required columns
# and `locations` is the dictionary defining location bounds
'''extracted_data = extract_locations(N_2016, locations)

# Compute monthly averages for each location
monthly_averages = {}
for location, location_data in extracted_data.items():
    monthly_averages[location] = monthly_average_data(location_data)

# Example: Print monthly averages for a specific location
for location, avg_data in monthly_averages.items():
    print(f"Monthly Averages for {location}:\n", avg_data)'''


In [ ]:
extracted_data11 = extract_locations(N_2011, locations)
monthly_averages11 = {}
for location, location_data in extracted_data11.items():
    monthly_averages11[location] = monthly_average_data(location_data)


monthly_averages11

In [ ]:
extracted_data06 = extract_locations(N_2006, locations)
monthly_averages06 = {}
for location, location_data in extracted_data06.items():
    monthly_averages06[location] = monthly_average_data(location_data)


monthly_averages06

In [ ]:
extracted_data20 = extract_locations(NA_2020, locations)
monthly_averages20 = {}
for location, location_data in extracted_data20.items():
    monthly_averages20[location] = monthly_average_data(location_data)


monthly_averages20

extracted_data21 = extract_locations(N_2021, locations)
monthly_averages21 = {}
for location, location_data in extracted_data21.items():
    monthly_averages21[location] = monthly_average_data(location_data)


monthly_averages21

In [ ]:
def plot_multiple_years(locations_dicts, years, location_names):
    # Filter location names based on availability in dictionaries
    available_locations = [
        location for location in location_names 
        if any(location in data_dict for data_dict in locations_dicts)
    ]
    
    # Set the number of subplots to 1 row and as many columns as there are locations
    num_locations = len(available_locations)
    rows, cols = 1, num_locations  # 1x2 layout for two locations
    
    # Create subplots
    fig, axes = plt.subplots(rows, cols, figsize=(18, 6), dpi=300, sharey=True)
    axes = axes if num_locations > 1 else [axes]  # Ensure axes is iterable even if there's one subplot
    
    # Colors for the lines
    colors = ['red', 'blue', 'green', 'orange', 'k']
    
    # Iterate over available locations
    for i, (ax, location) in enumerate(zip(axes, available_locations)):
        # Plot data from each dictionary for this location
        for j, (data_dict, year) in enumerate(zip(locations_dicts, years)):
            if location in data_dict:
                monthly_df = data_dict[location]
                ax.plot(
                    monthly_df['Month'], monthly_df['Sample Measurement'],
                    label=year, color=colors[j], marker='o', linestyle='-'
                )
        
        # Add title for the subplot
        ax.set_title(location, fontsize=20, fontweight='bold')
        
        # Customize x-axis labels and ticks
        ax.set_xlabel('Month', fontsize=20)
        ax.tick_params(axis='x', labelsize=20)
        ax.set_xticks(range(1, 13))  # Set consistent x-axis ticks (Months 1–12)
        ax.set_xticklabels(
            ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
            fontsize=20
        )
        
        # Y-axis labels: Only on the first subplot
        if i == 0:
            ax.set_ylabel('NO$_2$ (ppb)', fontsize=20)
            ax.tick_params(axis='y', labelsize=20)
        else:
            ax.tick_params(axis='y', labelleft=False)  # Hide y-axis ticks for other subplots
        
        # Add grid
        ax.grid(True, linestyle='--', alpha=0.6)
        
        # Add legend only for the first subplot
        if i == 0:
            ax.legend(title='Year', title_fontsize=20, fontsize=20)
    
    # Adjust layout
    plt.tight_layout()
    plt.show()

# Example usage with the 3 dictionaries
locations_dicts = [monthly_averages11, monthly_averages20, monthly_averages21]
years = ['2010-2019', '2020', '2021-2022']

# List of all possible locations (only two have data: 'Childrens Park' and 'Craycroft')
location_names = ['Children\'s Park', 'Craycroft']

# Plot the data
plot_multiple_years(locations_dicts, years, location_names)


In [ ]:
import matplotlib.gridspec as gridspec

# ---- TOP TWO SUBPLOTS (Monthly Averages) ----
def plot_top_two_subplots(locations_dicts, years, location_names, fig, gs):
    colors = ['red', 'blue', 'green', 'orange', 'k']
    axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])]
    
    for i, (ax, location) in enumerate(zip(axes, location_names)):
        for j, (data_dict, year) in enumerate(zip(locations_dicts, years)):
            if location in data_dict:
                monthly_df = data_dict[location]
                ax.plot(
                    monthly_df['Month'], monthly_df['Sample Measurement'],
                    label=year, color=colors[j], marker='o', linestyle='-'
                )
        ax.set_title(location, fontsize=20, fontweight='bold')
        ax.set_xlabel('Month', fontsize=20)
        ax.set_xticks(range(1, 13))
        ax.set_xticklabels(
            ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
             'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
            fontsize=20
        )
        if i == 0:
            ax.set_ylabel('NO$_2$ (ppb)', fontsize=20)
            ax.tick_params(axis='y', labelsize=20)
            ax.legend(title='Year', title_fontsize=20, fontsize=20)
        else:
            ax.tick_params(axis='y', labelleft=False)
        ax.tick_params(axis='x', labelsize=20)
        ax.grid(True, linestyle='--', alpha=0.6)

# ---- BOTTOM SUBPLOT (Yearly Medians) ----
def plot_bottom_subplot(yearly_medians, fig, gs):
    ax = fig.add_subplot(gs[1, :])
    for location, df in yearly_medians.items():
        if not df.empty:
            ax.plot(df['Year'], df['Sample Measurement'], label=location, marker='o')
    ax.set_ylabel('NO$_2$ (ppb)', fontsize=20)
    ax.set_xlabel('Year', fontsize=20)
    ax.tick_params(axis='both', labelsize=20)
    ax.minorticks_on()
    ax.tick_params(axis='x', which='minor', length=4, color='gray')
    ax.grid(True)
    ax.legend(fontsize=20)

# ---- FINAL COMBINED PLOT ----
fig = plt.figure(figsize=(18, 12), dpi=300)
gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1])  # 2 rows, 2 columns

# Data for the top plots
locations_dicts = [monthly_averages11, monthly_averages20, monthly_averages21]
years = ['2010-2019', '2020', '2021-2022']
location_names = ['Children\'s Park', 'Craycroft']

# Data for the bottom plot
# Make sure NO2, extract_locations, and yearly_median_data are defined elsewhere
locations = {
    "Children's Park": (-110.9823, -110.9823, 32.29515, 32.29515),
    "Craycroft": (-110.878067, -110.878067, 32.204411, 32.204411)
}
extracted_data = extract_locations(NO2, locations)
yearly_medians = {loc: yearly_median_data(df) for loc, df in extracted_data.items()}

# Plot top and bottom
plot_top_two_subplots(locations_dicts, years, location_names, fig, gs)
plot_bottom_subplot(yearly_medians, fig, gs)

# Final layout
plt.tight_layout()
plt.show()


In [ ]:
# Define locations
locations = {
    "Children's Park": (-110.9823, 32.29515),
    "Craycroft": (-110.878067, 32.204411)
}

# Define seasons
seasons = ['Fall/Winter', 'Spring', 'Dry Summer', 'Monsoon Summer']

# Define plot colors
colors = {
    "Children's Park": 'red',
    "Craycroft": 'blue'
}

# Extract data by site
def extract_data_by_site(data, locations):
    site_data = {}
    for site, (lon, lat) in locations.items():
        df = data[(data['Longitude'] == lon) & (data['Latitude'] == lat)].copy()
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])
        site_data[site] = df
    return site_data

# Filter by season
def seasonal_aggregation(df, season):
    df['Month'] = df['Timestamp'].dt.month
    if season == 'Fall/Winter':
        return df[df['Month'].isin([9, 10, 11, 12, 1, 2])]
    elif season == 'Spring':
        return df[df['Month'].isin([3, 4, 5])]
    elif season == 'Dry Summer':
        return df[df['Month'] == 6]
    elif season == 'Monsoon Summer':
        return df[df['Month'].isin([7, 8])]
    return df

# Plot only 2020 data
def plot_2020_seasonal_cycle(site_data):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharey=True, sharex=True)
    day_order = ['Saturday', 'Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

    for i, season in enumerate(seasons):
        ax = axes.flat[i]  # fix: flatten axes array
        for site, df in site_data.items():
            df_2020 = df[df['Timestamp'].dt.year == 2020].copy()
            season_df = seasonal_aggregation(df_2020, season)
            if not season_df.empty:
                season_df['Day'] = pd.Categorical(season_df['Timestamp'].dt.day_name(), categories=day_order, ordered=True)
                median = season_df.groupby('Day')['Sample Measurement'].median()
                q25 = season_df.groupby('Day')['Sample Measurement'].quantile(0.25)
                q75 = season_df.groupby('Day')['Sample Measurement'].quantile(0.75)
                ax.plot(median.index, median, marker='o', label=site, color=colors[site])
                ax.plot(q25.index, q25, linestyle='--', color=colors[site], alpha=0.5)
                ax.plot(q75.index, q75, linestyle='--', color=colors[site], alpha=0.5)
                ax.tick_params(axis='both', labelsize=18)
        ax.set_title(f'{season}', fontsize=18)
        
        #ax.set_xlabel('Day of the Week', fontsize=18)
        ax.set_xticklabels(day_order, rotation=45)
        ax.grid(True)
        #ax.set_ylabel('NO$_2$ (ppb)', fontsize=18)
        # Set xlabel only on bottom row (i.e., index 2 or 3)
        if i == 1:
            ax.legend(loc='upper right', fontsize=16)

        if i in [2, 3]:
            ax.set_xlabel('Day of the Week', fontsize=18)

        # Set ylabel only on left column (i.e., index 0 or 2)
        if i in [0, 2]:
            ax.set_ylabel('NO$_2$ (ppb)', fontsize=18)

    plt.tight_layout()
    plt.show()

# Main Execution
site_data = extract_data_by_site(NO2, locations)
plot_2020_seasonal_cycle(site_data)
